# popcoord Demo

This notebook demonstrates how to use the `popcoord` package to query population statistics, demographics, and density for any location on Earth.

**Only `lat` and `lon` are required** — everything else has sensible defaults:
- `radius_km = 5.0` (5 km search circle)
- `year` = latest year available for the chosen backend
- `backend = "api"` for `population()` and `demographics()`, `"raster"` for `density()`

## Installation

```bash
pip install popcoord            # API backend (no extra deps)
pip install popcoord[raster]    # + WorldPop raster + JRC GHS-POP backends (adds rasterio)
```


## Import the Package

In [1]:
import popcoord

## Example 1: Total Population Query

Get the total population within a radius around a coordinate (Amsterdam).
Only `lat` and `lon` are required.


In [2]:
# Minimal call — only lat and lon required
pop = popcoord.population(52.38, 4.90)

print(pop)
print(f"\nTotal population: {pop.total:,.0f} people")
print(f"(radius: {pop.radius_km} km, year: {pop.year}, backend: {pop.backend!r})")

# You can also be explicit with any parameter:
# pop = popcoord.population(52.38, 4.90, radius_km=10, year=2019, backend="api")


PopulationResult(total=433,424, year=2020, radius_km=5.0, backend='api')

Total population: 433,424 people
(radius: 5.0 km, year: 2020, backend: 'api')


## Example 2: Demographics Breakdown

Get detailed age and sex demographics for London.


In [3]:
# Demographics within 15 km of London (2020, WorldPop API)
demo = popcoord.demographics(
    lat=51.51,
    lon=-0.13,
    radius_km=15,
    year=2020
)

print(demo)
print(f"\nTotal: {demo.total:,.0f}")
print(f"Male: {demo.male:,.0f}")
print(f"Female: {demo.female:,.0f}")
print(f"Sex ratio (M/F): {demo.sex_ratio:.3f}")
print(f"Dependency ratio: {demo.dependency_ratio:.3f}")
print(f"Median age bucket: {demo.median_age_bucket}")


DemographicResult(total=6,137,268, male=3,026,086, female=3,111,183, age_groups=18, year=2020)

Total: 6,137,268
Male: 3,026,086
Female: 3,111,183
Sex ratio (M/F): 0.973
Dependency ratio: 0.569
Median age bucket: 45_49


### Access Individual Age Groups

In [4]:
# Print first few age groups
for label in ['0_1', '1_4', '5_9', '20_24', '65_69', '80_plus']:
    ag = demo.age_groups[label]
    print(f"{label:>8s}: total={ag.total:>10,.0f}, male={ag.male:>10,.0f}, female={ag.female:>10,.0f}")

     0_1: total=    71,684, male=    36,676, female=    35,008
     1_4: total=   287,703, male=   147,253, female=   140,449
     5_9: total=   369,022, male=   188,684, female=   180,338
   20_24: total=   394,392, male=   194,235, female=   200,157
   65_69: total=   306,145, male=   144,775, female=   161,370
 80_plus: total=   312,007, male=   125,320, female=   186,687


### Full Summary Report

In [5]:
print(demo.summary())

Demographics — 15 km around (51.51, -0.13), year 2020
  Total population :    6,137,268
  Male             :    3,026,086
  Female           :    3,111,183
  Sex ratio (M/F)  :        0.973
  Dependency ratio :        0.569
  Median age bucket:        45_49
  Age groups:
         0_1:     71,684  (m=    36,676  f=    35,008)    1.2%
         1_4:    287,703  (m=   147,253  f=   140,449)    4.7%
       10_14:    357,844  (m=   182,077  f=   175,767)    5.8%
       15_19:    330,539  (m=   169,351  f=   161,189)    5.4%
       20_24:    394,392  (m=   194,235  f=   200,157)    6.4%
       25_29:    425,997  (m=   210,092  f=   215,905)    6.9%
       30_34:    419,286  (m=   216,495  f=   202,791)    6.8%
       35_39:    404,940  (m=   210,548  f=   194,392)    6.6%
       40_44:    369,925  (m=   189,823  f=   180,103)    6.0%
       45_49:    395,171  (m=   199,402  f=   195,769)    6.4%
         5_9:    369,022  (m=   188,684  f=   180,338)    6.0%
       50_54:    420,684  (m=   207

## Example 3: Population Density

`density()` returns mean/max/min persons per km².
The `"raster"` backend (default) reads individual 1 km² pixels from a COG and gives a true spatial spread.
Here we use `backend="api"` for reliability in the notebook — it returns mean density (total ÷ circle area).

In [7]:
# Density within 20 km of NYC (API backend — reliable in notebooks)
# Use backend="raster" for pixel-level min/max/mean (requires rasterio + COG server access)
density = popcoord.density(
    lat=40.71,
    lon=-74.01,
    radius_km=20,
    year=2020,
    backend="api",
)

print(density)
print(f"\nMean density  : {density.mean_density:,.1f} people/km²")
print(f"Total pop     : {density.total_population:,.0f}")
print(f"Search area   : {density.area_km2:,.1f} km²")

DensityResult(mean=7,061.1 ppl/km², total=8,873,288, year=2020)

Mean density  : 7,061.1 people/km²
Total pop     : 8,873,288
Search area   : 1,256.6 km²


## Example 4: Comparing Different Locations

Let's compare population statistics across multiple major cities.

In [8]:
cities = {
    "Paris": (48.86, 2.35),
    "Tokyo": (35.68, 139.65),
    "Mumbai": (19.08, 72.88),
    "São Paulo": (-23.55, -46.63),
    "Cairo": (30.04, 31.24)
}

radius = 10  # km
year = 2020

print(f"Population within {radius} km radius (year {year}):\n")
print(f"{'City':<15} {'Latitude':>10} {'Longitude':>10} {'Population':>15}")
print("-" * 55)

for city_name, (lat, lon) in cities.items():
    pop = popcoord.population(lat=lat, lon=lon, radius_km=radius, year=year)
    print(f"{city_name:<15} {lat:>10.2f} {lon:>10.2f} {pop.total:>15,.0f}")

Population within 10 km radius (year 2020):

City              Latitude  Longitude      Population
-------------------------------------------------------
Paris                48.86       2.35       4,984,694
Tokyo                35.68     139.65       5,228,372
Mumbai               19.08      72.88       8,308,736
São Paulo           -23.55     -46.63       3,706,060
Cairo                30.04      31.24      10,141,084


## Example 5: Historical Population (GHS-POP, back to 1975)

The `"ghspop"` backend (JRC Global Human Settlement Layer) provides population estimates
at 5-year epochs from **1975 to 2030**. Any year is automatically snapped to the nearest epoch.
2025 and 2030 are modelled projections; 1975–2020 are calibrated estimates.


In [9]:
# Beijing population 1975–2020 using GHS-POP (5-year epochs)
beijing = (39.90, 116.40)
radius_km = 25

print(f"Population within {radius_km} km of Beijing (JRC GHS-POP, 5-year epochs):\n")
print(f"{'Year':>6} {'Population':>15} {'Change':>12}")
print("-" * 38)

prev_pop = None
for year in range(1975, 2021, 5):
    pop = popcoord.population(
        lat=beijing[0],
        lon=beijing[1],
        radius_km=radius_km,
        year=year,
        backend="ghspop"
    )

    change = ""
    if prev_pop is not None:
        pct_change = ((pop.total - prev_pop) / prev_pop) * 100
        direction = "▲" if pct_change >= 0 else "▼"
        change = f"{direction} {abs(pct_change):>5.1f}%"

    print(f"{pop.year:>6} {pop.total:>15,.0f} {change:>12}")
    prev_pop = pop.total


Population within 25 km of Beijing (JRC GHS-POP, 5-year epochs):

  Year      Population       Change
--------------------------------------
  1975       5,735,135             
  1980       6,152,088     ▲   7.3%
  1985       6,641,024     ▲   7.9%
  1990       7,196,188     ▲   8.4%
  1995       8,475,216     ▲  17.8%
  2000       9,946,109     ▲  17.4%
  2005      11,944,811     ▲  20.1%
  2010      14,336,521     ▲  20.0%
  2015      15,223,762     ▲   6.2%
  2020      16,218,935     ▲   6.5%


## Example 6: Comparing Backends

Three backends are available. They use different data sources and have different year ranges:

| Backend | Year Range | Notes |
| --- | --- | --- |
| `"api"` (default) | 2000–2020 | WorldPop REST API — no extra deps |
| `"raster"` | 2000–2022 | WorldPop COG rasters — requires `rasterio` |
| `"ghspop"` | **1975–2030** | JRC GHS-POP — requires `rasterio` |

> For the raster backends (`"raster"` and `"ghspop"`), install with: `pip install popcoord[raster]`


In [10]:
# Compare WorldPop API and GHS-POP for the same query (Sydney, 2020)
lat, lon = -33.87, 151.21
radius_km = 15

# API backend (WorldPop)
pop_api = popcoord.population(lat=lat, lon=lon, radius_km=radius_km, year=2020, backend="api")
print(f"WorldPop API:  {pop_api.total:,.0f} people  ({pop_api.source})")

# GHS-POP backend (JRC) — note: year 2020 is available in both
pop_ghspop = popcoord.population(lat=lat, lon=lon, radius_km=radius_km, year=2020, backend="ghspop")
print(f"JRC GHS-POP:   {pop_ghspop.total:,.0f} people  ({pop_ghspop.source})")

diff_pct = abs(pop_api.total - pop_ghspop.total) / pop_api.total * 100
print(f"\nDifference: {diff_pct:.1f}% — expected; datasets use different methodologies")


WorldPop API:  1,539,468 people  (WorldPop REST API (wpgppop))
JRC GHS-POP:   1,861,352 people  (JRC GHS-POP R2023A (epoch 2020, 30 arc-sec / ~1 km))

Difference: 20.9% — expected; datasets use different methodologies


## Example 7: Exploring All Age Groups

In [11]:
# Get demographics for Mexico City
demo = popcoord.demographics(
    lat=19.43, lon=-99.13, radius_km=20, year=2020
)

# Create a simple bar chart of population by age group
print("Population by Age Group (Mexico City, 20 km radius):\n")

max_pop = max(ag.total for ag in demo.age_groups.values())

for label, ag in demo.age_groups.items():
    bar_length = int((ag.total / max_pop) * 40)
    bar = "█" * bar_length
    pct = (ag.total / demo.total) * 100
    print(f"{label:>8s} {bar:<40s} {ag.total:>10,.0f} ({pct:>4.1f}%)")

Population by Age Group (Mexico City, 20 km radius):

     0_1 █████                                       752,460 ( 1.2%)
     1_4 ████████████████████████                  3,232,581 ( 5.2%)
   10_14 █████████████████████████████████         4,373,092 ( 7.0%)
   15_19 ████████████████████████████████████      4,793,267 ( 7.7%)
   20_24 ███████████████████████████████████████   5,150,118 ( 8.3%)
   25_29 ████████████████████████████████████████  5,264,001 ( 8.5%)
   30_34 ██████████████████████████████████████    5,122,478 ( 8.3%)
   35_39 ███████████████████████████████████       4,706,899 ( 7.6%)
   40_44 ████████████████████████████████████      4,779,156 ( 7.7%)
   45_49 ███████████████████████████████████       4,692,401 ( 7.6%)
     5_9 █████████████████████████████████         4,351,231 ( 7.0%)
   50_54 ████████████████████████████              3,762,838 ( 6.1%)
   55_59 ████████████████████████                  3,167,212 ( 5.1%)
   60_64 ████████████████████                    

## Parameter Reference

### Required Parameters

Only `lat` and `lon` are required. Pass them positionally or as keyword arguments:

```python
popcoord.population(52.37, 4.90)                          # positional
popcoord.population(lat=52.37, lon=4.90)                  # keyword
popcoord.population(52.37, 4.90, radius_km=10, year=2015) # override defaults
```

### Optional Parameters (with defaults)

| Parameter | Type | Default | Valid Range | Description |
| --- | --- | --- | --- | --- |
| `lat` | `float` | **required** | `-90` to `90` | Latitude (WGS-84 decimal degrees) |
| `lon` | `float` | **required** | `-180` to `180` | Longitude (WGS-84 decimal degrees) |
| `radius_km` | `float` | `5.0` | `> 0` | Search radius in kilometres |
| `year` | `int` | latest for backend | see below | Reference year; clamped/snapped automatically |
| `backend` | `str` | `"api"` †  | `"api"`, `"raster"`, `"ghspop"` | Data source backend |

† `density()` defaults to `"raster"` for pixel-level min/max/mean.

### Year Range by Function and Backend

| Function | `"api"` | `"raster"` | `"ghspop"` |
| --- | --- | --- | --- |
| `population()` | 2000–2020 | 2000–2022 ‡ | **1975–2030** § |
| `demographics()` | 2000–2020 | 2000–2020 | not supported |
| `density()` | 2000–2020 | 2000–2022 ‡ | **1975–2030** § |

‡ 2021–2022 are UN-adjusted mosaics; age-sex data at these years uses a different schema.  
§ GHS-POP epochs every 5 years; any year is snapped to nearest epoch. 2025/2030 are projections.

### Available Age Groups (for `demographics()`)

`0_1`, `1_4`, `5_9`, `10_14`, `15_19`, `20_24`, `25_29`, `30_34`, `35_39`, `40_44`, `45_49`, `50_54`, `55_59`, `60_64`, `65_69`, `70_74`, `75_79`, `80_plus`


## Summary

`popcoord` provides a minimal, research-friendly interface to global population data:

- **Only `lat` and `lon` are required.** Radius defaults to 5 km; year defaults to the latest available for the chosen backend.
- **Three backends**, same interface:
  1. `"api"` — WorldPop REST API, 2000–2020, no extra deps
  2. `"raster"` — WorldPop COG rasters, 2000–2022, requires `rasterio`
  3. `"ghspop"` — JRC GHS-POP, **1975–2030**, requires `rasterio`
- **Three functions:**
  - `population()` — total headcount (all 3 backends)
  - `demographics()` — 18 age groups × male/female (WorldPop only; 2000–2020)
  - `density()` — persons per km² (all 3 backends)

### Data sources

- [WorldPop](https://www.worldpop.org/) — CC BY 4.0, ~1 km global resolution
- [JRC GHS-POP R2023A](https://ghsl.jrc.ec.europa.eu/) — European Commission open data, ~1 km global resolution
